In [18]:
import torch
from torch import nn
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from PIL import Image
from torchvision.transforms import v2
import pandas as pd
import numpy as np
import os

**Enable cuda if available**

In [19]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [20]:
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [21]:
writer = SummaryWriter()

In [22]:
class ISIC2019(Dataset): # TODO: consider maybe removing downsampled or removing duplicates. unsure if these are necessary
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.ohe_labels = self.img_labels.iloc[:, 1:]
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f"{self.img_labels.iloc[idx, 0]}.jpg")
        image = Image.open(img_path)
        label = np.where(self.ohe_labels==1)[1][idx]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

In [23]:
transform = v2.Compose([
    v2.Resize((224, 224)),
    v2.ToTensor(),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

C:\Users\abhin\AppData\Roaming\Python\Python312\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [24]:
isic_dataset = ISIC2019(img_dir='../data/ISIC_2019_Training_Input', annotations_file='../data/ISIC_2019_Training_GroundTruth.csv', transform=transform)
train_images_size = len(isic_dataset)

In [25]:
train_size = int(train_images_size * 0.80)
test_size = train_images_size - train_size
# 80% train 20% test

isic_train, isic_test = random_split(isic_dataset, [train_size, test_size])
train_size, test_size

(20264, 5067)

In [26]:
class MobileNetV3(nn.Module): # unused for now
    def __init__(self):
        super().__init__()
        self.model =  nn.Sequential(nn.Conv2d(1,6,5, padding=2),
                                    nn.Sigmoid(),
                                    nn.AvgPool2d(2, stride=2),
                                    nn.Conv2d(6,16,5),
                                    nn.Sigmoid(),
                                    nn.AvgPool2d(2, stride=2),
                                    nn.Flatten(),
                                    nn.Linear(400, 120),
                                    nn.Sigmoid(),
                                    nn.Linear(120, 84),
                                    nn.Sigmoid(),
                                    nn.Linear(84, 10))

    def forward(self, x):
        x = self.model(x)
        return x

In [27]:
num_classes = 9

mobilenet = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT) # use imagenet pretrained weights
in_features = mobilenet.classifier[3].in_features
mobilenet.classifier[3] = nn.Linear(in_features, num_classes) # change number of output categories

nn.init.normal_(mobilenet.classifier[3].weight, 0, 0.01) 
nn.init.zeros_(mobilenet.classifier[3].bias) # use same weight + bias init as pytorch for consistency

mobilenet

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bi

**Move model to specified device**

In [28]:
mobilenet = mobilenet.to(device=device)

In [ ]:
epochs = 100
learning_rate = 1e-4
batch_size = 32

# TODO - find better parameters for baseline

In [30]:
dataloader_train = DataLoader(isic_train, batch_size=batch_size, shuffle=True)
dataloader_test = DataLoader(isic_test, batch_size=batch_size, shuffle=True)

num_train_batches = len(dataloader_train)
num_test_batches = len(dataloader_test)

loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mobilenet.parameters(), lr=learning_rate)
num_train_batches, num_test_batches

(1267, 317)

In [ ]:
for epoch in range(epochs):
    train_loss = 0
    train_acc = 0

    mobilenet.train()
    for batch_idx, (train_features, train_labels) in enumerate(dataloader_train):
        print(batch_idx)
        train_features = train_features.to(device)
        train_labels = train_labels.to(device) # move to device

        optimizer.zero_grad()

        predictions = mobilenet(train_features)        
        predictions_labels = torch.argmax(predictions, dim=1)
        
        train_batch_acc = (predictions_labels == train_labels).sum().item() / train_features.shape[0]

        train_batch_loss = loss(predictions, train_labels)
        train_batch_loss.backward()

        optimizer.step()

        train_loss += train_batch_loss.item()
        train_acc += train_batch_acc

    val_loss = 0
    val_acc = 0

    mobilenet.eval()
    with torch.no_grad():
        for batch_idx, (test_features, test_labels) in enumerate(dataloader_test):
            test_features = test_features.to(device)
            test_labels = test_labels.to(device) # move to device
            
            predictions = mobilenet(test_features)
            predictions_labels = torch.argmax(predictions, dim=1)

            test_batch_acc = (predictions_labels == test_labels).sum().item() / test_features.shape[0]
            test_batch_loss = loss(predictions, test_labels)

            val_loss += test_batch_loss.item()
            val_acc += test_batch_acc

    train_loss /= num_train_batches
    train_acc /= num_train_batches

    val_loss /= num_test_batches
    val_acc /= num_test_batches

    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_acc, epoch)

    writer.add_scalar("Loss/val", val_loss, epoch)
    writer.add_scalar('Accuracy/val', val_acc, epoch)

In [ ]:
writer.flush()